In [ ]:
%pip install -r requirements.txt

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [5]:
df = pd.read_csv("../Data/crop_weather_soil.csv")
df.shape 

(577685, 20)

In [6]:
df.columns

Index(['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area',
       'Production', 'yield', 'Avg_Temperature', 'Avg_Humidity',
       'Total_Rainfall', 'merge_key_x', 'PH', 'Clay', 'Sand', 'Silt',
       'Nitrogen', 'SOC', 'CEC', 'merge_key_y'],
      dtype='str')

In [7]:
df.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield,Avg_Temperature,Avg_Humidity,Total_Rainfall,merge_key_x,PH,Clay,Sand,Silt,Nitrogen,SOC,CEC,merge_key_y
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,1.594896,27.57,79.98,2356.32,ANDAMAN AND NICOBAR ISLANDS|NICOBARS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,0.500000,27.57,79.98,2356.32,ANDAMAN AND NICOBAR ISLANDS|NICOBARS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Rice,102.0,321.0,3.147059,27.57,79.98,2356.32,ANDAMAN AND NICOBAR ISLANDS|NICOBARS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Whole Year,Banana,176.0,641.0,3.642045,27.57,79.98,2356.32,ANDAMAN AND NICOBAR ISLANDS|NICOBARS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,0.229167,27.57,79.98,2356.32,ANDAMAN AND NICOBAR ISLANDS|NICOBARS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.dtypes

State_Name             str
District_Name          str
Crop_Year            int64
Season                 str
Crop                   str
Area               float64
Production         float64
yield              float64
Avg_Temperature    float64
Avg_Humidity       float64
Total_Rainfall     float64
merge_key_x            str
PH                 float64
Clay               float64
Sand               float64
Silt               float64
Nitrogen           float64
SOC                float64
CEC                float64
merge_key_y            str
dtype: object

In [8]:
soil_cols = [
    "PH",
    "Clay",
    "Sand",
    "Silt",
    "Nitrogen",
    "SOC",
    "CEC"
]

df[soil_cols].isnull().mean() * 100

PH          12.134641
Clay        12.134641
Sand        12.134641
Silt        12.134641
Nitrogen    12.134641
SOC         12.134641
CEC         12.134641
dtype: float64

In [10]:
crop_counts = df["Crop"].value_counts()
print("Unique crops:", df["Crop"].nunique())
crop_counts.head(20)

Unique crops: 77


Crop
Rice                     36310
Maize                    34026
Moong(Green Gram)        24448
Urad                     23884
Sesamum                  21219
Groundnut                20944
Wheat                    19152
Sugarcane                18590
Rapeseed &Mustard        18325
Arhar/Tur                18162
Potato                   17731
Onion                    17663
Gram                     17398
Jowar                    16603
Dry chillies             15276
Bajra                    13080
Sunflower                12463
Peas & beans (Pulses)    11602
Small millets            11580
Cotton(lint)             10652
Name: count, dtype: int64

In [19]:
crop_counts.describe()
# plt.figure(figsize=(12,5))
# sns.barplot(x=df["Crop"].head(20),y=df["Crop"].value_counts().head(20).values)
# plt.show()
# crop_counts.head(30).plot(kind="bar")

count       77.000000
mean      7502.402597
std       8184.531806
min        112.000000
25%        483.000000
50%       4623.000000
75%      10652.000000
max      36310.000000
Name: count, dtype: float64

In [18]:
feature_cols = [
    'State_Name',
    'District_Name',
    'Crop_Year',
    'Season',
    'Avg_Temperature',
    'Avg_Humidity',
    'Total_Rainfall',
    'PH',
    'Clay',
    'Sand',
    'Silt',
    'Nitrogen',
    'SOC',
    'CEC'
]
duplicates = df.duplicated(feature_cols).sum()

print(duplicates)

526487


In [24]:
district_crops_coount =( df.groupby(
    ["State_Name","District_Name"]
)["Crop"].nunique())
district_crops_coount.describe()


count    732.000000
mean      34.642077
std       12.499205
min        1.000000
25%       27.000000
50%       35.000000
75%       41.000000
max       64.000000
Name: Crop, dtype: float64

1. Model has duplicate values - which is common in this current dataset -  same state,district,year -  has many crops - multilabel recommecndation problem
2. District supports many crops
3. District-Year-Season ->List of Crops
4. Missing soil data percent is around 12% - pretty good -  can go ahead with Imputation
5. High class imbalance - need to consider f1_score not only accuracy



In [28]:
df["Crop_Year"].describe()

count    577685.000000
mean       2007.568301
std           6.170410
min        1997.000000
25%        2002.000000
50%        2007.000000
75%        2013.000000
max        2019.000000
Name: Crop_Year, dtype: float64

In [29]:
df["Crop_Year"].value_counts().sort_index()

Crop_Year
1997    17201
1998    23058
1999    24915
2000    26685
2001    26181
2002    29591
2003    30615
2004    27597
2005    27167
2006    28190
2007    28517
2008    28849
2009    28584
2010    28297
2011    29475
2012    28046
2013    29189
2014    25989
2015    16890
2016    17402
2017    17923
2018    18189
2019    19135
Name: count, dtype: int64

In [10]:
group_cols = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season"
]

grouped = (
    df.groupby(group_cols)["Crop"]
      .nunique()
)

grouped.describe()

count    50320.000000
mean         7.040322
std          5.080006
min          1.000000
25%          2.000000
50%          6.000000
75%         10.000000
max         36.000000
Name: Crop, dtype: float64

In [27]:
(grouped > 1).mean() * 100

np.float64(83.81756756756756)

In [11]:
agg_dict = {
    "Avg_Temperature": "first",
    "Avg_Humidity": "first",
    "Total_Rainfall": "first",
    "PH": "first",
    "Clay": "first",
    "Sand": "first",
    "Silt": "first",
    "Nitrogen": "first",
    "SOC": "first",
    "CEC": "first",
    "Crop": list
}
multilabel_df = (
    df.groupby(group_cols)
      .agg(agg_dict)
      .reset_index()
)

print(multilabel_df.shape)
multilabel_df.head()

(50320, 15)


,State_Name,District_Name,Crop_Year,Season,Avg_Temperature,Avg_Humidity,Total_Rainfall,PH,Clay,Sand,Silt,Nitrogen,SOC,CEC,Crop
0,ANDAMAN AND NICOBAR ISLANDS,ANDAMAN AND NICOBAR ISLANDS,2007,Kharif,27.86,78.38,2373.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[Arecanut, Banana, Black pepper, Dry chillies,..."
1,ANDAMAN AND NICOBAR ISLANDS,ANDAMAN AND NICOBAR ISLANDS,2007,Rabi,27.86,78.38,2373.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[Arecanut, Arhar/Tur, Banana, Black pepper, Dr..."
2,ANDAMAN AND NICOBAR ISLANDS,ANDAMAN AND NICOBAR ISLANDS,2007,Whole Year,27.86,78.38,2373.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[Coconut]
3,ANDAMAN AND NICOBAR ISLANDS,ANDAMAN AND NICOBAR ISLANDS,2008,Autumn,27.94,79.09,3189.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[Arecanut, Banana, Black pepper, Rice, Sugarcane]"
4,ANDAMAN AND NICOBAR ISLANDS,ANDAMAN AND NICOBAR ISLANDS,2008,Summer,27.94,79.09,3189.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[Arecanut, Arhar/Tur, Banana, Black pepper, Dr..."


In [12]:
features = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season",

    "Avg_Temperature",
    "Avg_Humidity",
    "Total_Rainfall",

    "PH",
    "Clay",
    "Sand",
    "Silt",
    "Nitrogen",
    "SOC",
    "CEC",
    "Crop"
]
#delete_columns = [col for col in df.columns.tolist() if col not in features]
delete_columns =['Area', 'Production', 'yield', 'merge_key_x', 'merge_key_y']
df = df.drop(columns = delete_columns)

In [13]:
crop_freq = (
    df["Crop"]
    .value_counts()
    .sort_values()
)
crop_freq.head(20)

Crop
Drum Stick                   112
Cauliflower                  122
Korra                        126
Grapes                       129
Beans & Mutter(Vegetable)    167
Total foodgrain              194
Cabbage                      204
Bhindi                       236
Pineapple                    247
Orange                       271
Pulses total                 272
Pome Fruit                   297
Citrus Fruit                 301
Tomato                       368
Other Vegetables             381
Brinjal                      386
Other Fresh Fruits           410
Mango                        449
Paddy                        479
Papaya                       483
Name: count, dtype: int64

In [14]:
multilabel_df["Crop"]

0        [Arecanut, Banana, Black pepper, Dry chillies,...
1        [Arecanut, Arhar/Tur, Banana, Black pepper, Dr...
2                                                [Coconut]
3        [Arecanut, Banana, Black pepper, Rice, Sugarcane]
4        [Arecanut, Arhar/Tur, Banana, Black pepper, Dr...
                               ...                        
50315    [Bajra, Mesta, Moong(Green Gram), Niger seed, ...
50316    [Arhar/Tur, Gram, Horse-gram, Khesari, Linseed...
50317    [Groundnut, Maize, Moong(Green Gram), Rice, Se...
50318                                 [Coconut, Sugarcane]
50319                                               [Rice]
Name: Crop, Length: 50320, dtype: object

In [15]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(multilabel_df["Crop"])
print(Y.shape)
print(len(mlb.classes_))

(50320, 77)
77


In [16]:
labels_per_row = Y.sum(axis=1)

print(labels_per_row.mean())
print(labels_per_row.min())
print(labels_per_row.max())

7.040321939586645
1
36


In [46]:
sorted(df["Crop"].unique())

['Arecanut',
 'Arhar/Tur',
 'Bajra',
 'Banana',
 'Barley',
 'Beans & Mutter(Vegetable)',
 'Bhindi',
 'Black pepper',
 'Brinjal',
 'Cabbage',
 'Cardamom',
 'Cashewnut',
 'Castor seed',
 'Cauliflower',
 'Citrus Fruit',
 'Coconut',
 'Coriander',
 'Cotton(lint)',
 'Cowpea(Lobia)',
 'Drum Stick',
 'Dry chillies',
 'Dry ginger',
 'Garlic',
 'Ginger',
 'Gram',
 'Grapes',
 'Groundnut',
 'Guar seed',
 'Horse-gram',
 'Jowar',
 'Jute',
 'Khesari',
 'Korra',
 'Linseed',
 'Maize',
 'Mango',
 'Masoor',
 'Mesta',
 'Moong(Green Gram)',
 'Moth',
 'Niger seed',
 'Oilseeds total',
 'Onion',
 'Orange',
 'Other  Rabi pulses',
 'Other Cereals',
 'Other Cereals & Millets',
 'Other Fresh Fruits',
 'Other Kharif pulses',
 'Other Rabi pulses',
 'Other Vegetables',
 'Paddy',
 'Papaya',
 'Peas & beans (Pulses)',
 'Pineapple',
 'Pome Fruit',
 'Potato',
 'Pulses total',
 'Ragi',
 'Rapeseed &Mustard',
 'Rice',
 'Safflower',
 'Sannhamp',
 'Sesamum',
 'Small millets',
 'Soyabean',
 'Sugarcane',
 'Sunflower',
 'Sweet p

In [17]:
INVALID_CROPS = [
    "Oilseeds total",
    "Pulses total",
    "Total foodgrain",
    "Other Cereals",
    "Other Cereals & Millets",
    "Other Fresh Fruits",
    "Other Vegetables",
    "Other Kharif pulses",
    "Other Rabi pulses",
    "Other  Rabi pulses",
    "other oilseeds"
]

In [18]:
multilabel_df[
    multilabel_df["Crop"].apply(
        lambda x: "Rice" in x and "Paddy" in x
    )
].shape

(354, 15)

In [19]:
def clean_crop_list(crops):
    return [
        crop
        for crop in crops
        if crop not in INVALID_CROPS
    ]
multilabel_df["Crop"] = (
    multilabel_df["Crop"]
    .apply(clean_crop_list)
)

In [20]:
multilabel_df = multilabel_df[
    multilabel_df["Crop"].str.len() > 0
]

In [21]:
#rebuilding multilabelbinarizer
mlb = MultiLabelBinarizer()

Y = mlb.fit_transform(
    multilabel_df["Crop"]
)
print(mlb.classes_)
print(Y.shape)
print(len(mlb.classes_))

['Arecanut' 'Arhar/Tur' 'Bajra' 'Banana' 'Barley'
 'Beans & Mutter(Vegetable)' 'Bhindi' 'Black pepper' 'Brinjal' 'Cabbage'
 'Cardamom' 'Cashewnut' 'Castor seed' 'Cauliflower' 'Citrus Fruit'
 'Coconut' 'Coriander' 'Cotton(lint)' 'Cowpea(Lobia)' 'Drum Stick'
 'Dry chillies' 'Dry ginger' 'Garlic' 'Ginger' 'Gram' 'Grapes' 'Groundnut'
 'Guar seed' 'Horse-gram' 'Jowar' 'Jute' 'Khesari' 'Korra' 'Linseed'
 'Maize' 'Mango' 'Masoor' 'Mesta' 'Moong(Green Gram)' 'Moth' 'Niger seed'
 'Onion' 'Orange' 'Paddy' 'Papaya' 'Peas & beans (Pulses)' 'Pineapple'
 'Pome Fruit' 'Potato' 'Ragi' 'Rapeseed &Mustard' 'Rice' 'Safflower'
 'Sannhamp' 'Sesamum' 'Small millets' 'Soyabean' 'Sugarcane' 'Sunflower'
 'Sweet potato' 'Tapioca' 'Tobacco' 'Tomato' 'Turmeric' 'Urad' 'Wheat']
(50197, 66)
66


1.Config data


In [22]:
FEATURE_COLUMNS = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season",
    "Avg_Temperature",
    "Avg_Humidity",
    "Total_Rainfall",
    "PH",
    "Clay",
    "Sand",
    "Silt",
    "Nitrogen",
    "SOC",
    "CEC"
]

TARGET_COLUMN = "Crop"
NUMERIC_FEATURES = [
    "Crop_Year",
    "Avg_Temperature",
    "Avg_Humidity",
    "Total_Rainfall",
    "PH",
    "Clay",
    "Sand",
    "Silt",
    "Nitrogen",
    "SOC",
    "CEC"
]
CATEGORICAL_FEATURES = [
    "State_Name",
    "District_Name",
    "Season"
]

### Data Split
1. Train : 1997-2015

2. Validation : 2016-2017

3. Test : 2018-2019

In [23]:
# splitting data
train_df = multilabel_df[
    multilabel_df["Crop_Year"] <=2015
]
valid_df = multilabel_df[
    (multilabel_df["Crop_Year"]>=2016)
    & 
    (multilabel_df["Crop_Year"]<=2017)
]
test_df = multilabel_df[
    multilabel_df["Crop_Year"] >= 2018
]
print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(40182, 15)
(4918, 15)
(5097, 15)


In [24]:
X_train = train_df[FEATURE_COLUMNS].copy(deep=True)
X_valid = valid_df[FEATURE_COLUMNS].copy(deep=True)
X_test = test_df[FEATURE_COLUMNS].copy(deep=True)
Y_train = mlb.transform(train_df["Crop"])
Y_valid = mlb.transform(valid_df["Crop"])
Y_test = mlb.transform(test_df["Crop"])

In [61]:
print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)

print(Y_train.shape)
print(Y_valid.shape)
print(Y_test.shape)

(40182, 14)
(4918, 14)
(5097, 14)
(40182, 66)
(4918, 66)
(5097, 66)


In [25]:
#label distribution
label_counts = pd.Series(
    Y_train.sum(axis=0),
    index=mlb.classes_
).sort_values()
print("rarestcrops:",label_counts.head(20))
print("common crops:",label_counts.tail(20))

rarestcrops: Drum Stick                    112
Cauliflower                   122
Korra                         126
Grapes                        129
Beans & Mutter(Vegetable)     167
Cabbage                       204
Bhindi                        236
Pineapple                     247
Orange                        271
Pome Fruit                    297
Citrus Fruit                  301
Tomato                        368
Cardamom                      384
Brinjal                       386
Mango                         449
Paddy                         479
Papaya                        483
Cowpea(Lobia)                 844
Moth                         1053
Black pepper                 1135
dtype: int64
common crops: Cotton(lint)              5060
Peas & beans (Pulses)     5391
Small millets             5694
Sunflower                 5948
Bajra                     6189
Dry chillies              7336
Jowar                     7823
Onion                     8182
Potato                    8333
G

In [26]:
numerical_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(handle_unknown="ignore"))
    ]
)
preprocessor=ColumnTransformer(
    transformers=[
        (
            "num",numerical_transformer,NUMERIC_FEATURES
        ),
        (
            "cat",categorical_transformer,CATEGORICAL_FEATURES
        )
    ]
) 

In [27]:
X_train_processed = preprocessor.fit_transform(X_train)
print(type(X_train_processed))
print(X_train_processed.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(40182, 718)


In [28]:
from sklearn.metrics import (
    f1_score,
    hamming_loss
)

In [29]:
def precision_at_k(y_true, y_prob, k=5):

    top_k = np.argsort(y_prob, axis=1)[:, -k:]

    scores = []

    for i in range(len(y_true)):
        true_labels = set(np.where(y_true[i] == 1)[0])
        pred_labels = set(top_k[i])

        scores.append(
            len(true_labels & pred_labels) / k
        )

    return np.mean(scores)
def evaluate_model(model, X, Y, threshold=0.5):

    Y_prob = model.predict_proba(X)

    Y_pred = (Y_prob >= threshold).astype(int)

    results = {
        "micro_f1":
            f1_score(
                Y,
                Y_pred,
                average="micro"
            ),

        "macro_f1":
            f1_score(
                Y,
                Y_pred,
                average="macro"
            ),

        "hamming_loss":
            hamming_loss(
                Y,
                Y_pred
            ),

        "precision_at_5":
            precision_at_k(
                Y,
                Y_prob,
                k=5
            )
    }

    return results

In [30]:
def recall_at_k(y_true, y_prob, k=5):

    top_k = np.argsort(y_prob, axis=1)[:, -k:]

    scores = []

    for i in range(len(y_true)):

        true_labels = set(
            np.where(y_true[i] == 1)[0]
        )

        pred_labels = set(
            top_k[i]
        )

        if len(true_labels) == 0:
            continue

        scores.append(
            len(true_labels & pred_labels)
            /
            len(true_labels)
        )

    return np.mean(scores)

In [31]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

In [32]:
lr_pipeline = Pipeline(
    steps=[
        ("preprocessor",preprocessor),
        ("classifier", OneVsRestClassifier(
            LogisticRegression(random_state=42,max_iter=2000,n_jobs=-1)
        ))
    ]
)
lr_pipeline.fit(X_train,Y_train)

d:\Agrisense Application\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please lea

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](66,)","[ 0, 1, 2,...,63,64,65]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['State_Name','District_Name','Crop_Year',...,'Nitrogen','SOC','CEC']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder

In [33]:
lr_results = evaluate_model(
    lr_pipeline,
    X_valid,
    Y_valid
)

print(lr_results)

{'micro_f1': 0.7674720551584163, 'macro_f1': 0.5389446317260711, 'hamming_loss': 0.04800239072300886, 'precision_at_5': np.float64(0.6903212688084587)}


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [34]:
import time
benchmark_results=[]
def benchmark_model(
    model_name,
    model,
    X_train,
    Y_train,
    X_valid,
    Y_valid
):
    start = time.time()

    model.fit(X_train, Y_train)

    training_time = time.time() - start

    metrics = evaluate_model(
        model,
        X_valid,
        Y_valid
    )

    result = {
        "Model": model_name,
        "Micro_F1": round(metrics["micro_f1"], 4),
        "Macro_F1": round(metrics["macro_f1"], 4),
        "Hamming_Loss": round(metrics["hamming_loss"], 4),
        "Precision@5": round(metrics["precision_at_5"], 4),
        "Training_Time_Sec": round(training_time, 2)
    }

    benchmark_results.append(result)

    return result

In [76]:
lr_result = benchmark_model(
    "Logistic Regression",
    lr_pipeline,
    X_train,
    Y_train,
    X_valid,
    Y_valid
)

print(lr_result)

d:\Agrisense Application\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please lea

{'Model': 'Logistic Regression', 'Micro_F1': 0.7675, 'Macro_F1': 0.5389, 'Hamming_Loss': 0.048, 'Precision@5': np.float64(0.6903), 'Training_Time_Sec': 23.37}


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [77]:
benchmark_df = pd.DataFrame(
    benchmark_results
)

benchmark_df.sort_values(
    "Micro_F1",
    ascending=False
)
benchmark_df.to_csv(
    "model_benchmark_results.csv",
    index=False
)

In [2]:
%pip install -q lightgbm catboost

Note: you may need to restart the kernel to use updated packages.


In [44]:
from sklearn.ensemble import ( RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
MODELS = {
    "RandomForest": OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    ),

    "ExtraTrees": OneVsRestClassifier(
        ExtraTreesClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    )
}
MODELS1 ={
    "XGBoost": OneVsRestClassifier(
        XGBClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
    ),

    "LightGBM": OneVsRestClassifier(
        LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose=-1,
            n_jobs=-1
        )
    ),

    "CatBoost": OneVsRestClassifier(
        CatBoostClassifier(
            iterations=200,
            learning_rate=0.1,
            depth=6,
            loss_function="Logloss",
            verbose=False,
            random_seed=42
        )
    )
}
# rf_pipeline=Pipeline(
#     steps=[
#         ("model",
#     OneVsRestClassifier(
#     RandomForestClassifier(
#         n_estimators=200,
#         random_state=42,
#         n_jobs=-1
#     )
#     )
#         )
#     ]
# )
# rf_result = benchmark_model(
#     "RandomForestClassifier",
#     rf_pipeline,
#      X_train,
#     Y_train,
#     X_valid,
#     Y_valid
# )

In [45]:
benchmark_results = []

trained_models = {}
def run_benchmark(
    models,
    preprocessor,
    X_train,
    Y_train,
    X_valid,
    Y_valid
):

    global benchmark_results
    global trained_models

    for model_name, classifier in models.items():

        print("=" * 80)
        print(f"Training: {model_name}")
        print("=" * 80)

        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("classifier", classifier)
            ]
        )

        start_time = time.time()

        pipeline.fit(
            X_train,
            Y_train
        )

        training_time = (
            time.time() - start_time
        )

        metrics = evaluate_model(
            pipeline,
            X_valid,
            Y_valid
        )

        result = {
            "Model": model_name,
            "Micro_F1": round(
                metrics["micro_f1"], 4
            ),
            "Macro_F1": round(
                metrics["macro_f1"], 4
            ),
            "Hamming_Loss": round(
                metrics["hamming_loss"], 4
            ),
            "Precision@5": round(
                metrics["precision_at_5"], 4
            ),
            "Training_Time_Sec": round(
                training_time, 2
            )
        }

        benchmark_results.append(
            result
        )

        trained_models[
            model_name
        ] = pipeline

        print(result)

    return pd.DataFrame(
        benchmark_results
    )
benchmark_df = run_benchmark(
    MODELS1,
    preprocessor,
    X_train,
    Y_train,
    X_valid,
    Y_valid
)

Training: XGBoost


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'Model': 'XGBoost', 'Micro_F1': 0.8538, 'Macro_F1': 0.619, 'Hamming_Loss': 0.0294, 'Precision@5': np.float64(0.7305), 'Training_Time_Sec': 184.34}
Training: LightGBM


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'Model': 'LightGBM', 'Micro_F1': 0.8626, 'Macro_F1': 0.6273, 'Hamming_Loss': 0.0279, 'Precision@5': np.float64(0.7334), 'Training_Time_Sec': 120.44}
Training: CatBoost


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'Model': 'CatBoost', 'Micro_F1': 0.8598, 'Macro_F1': 0.6278, 'Hamming_Loss': 0.0288, 'Precision@5': np.float64(0.7285), 'Training_Time_Sec': 2506.23}


In [46]:
benchmark_df = benchmark_df.sort_values(
    "Micro_F1",
    ascending=False
)

benchmark_df

,Model,Micro_F1,Macro_F1,Hamming_Loss,Precision@5,Training_Time_Sec
1,LightGBM,0.8626,0.6273,0.0279,0.7334,120.44
2,CatBoost,0.8598,0.6278,0.0288,0.7285,2506.23
0,XGBoost,0.8538,0.6190,0.0294,0.7305,184.34


In [38]:
benchmark_df.to_csv(
    "benchmark_results.csv",
    index=False
)

In [84]:
best_model_name = (
    benchmark_df.iloc[0]["Model"]
)

print(best_model_name)

ExtraTrees


In [ ]:
best_model = trained_models[
    best_model_name
]

In [39]:
FEATURE_COLUMNS_NO_GEO = [
    "Crop_Year",
    "Season",
    "Avg_Temperature",
    "Avg_Humidity",
    "Total_Rainfall",
    "PH",
    "Clay",
    "Sand",
    "Silt",
    "Nitrogen",
    "SOC",
    "CEC"
]

NUMERIC_FEATURES_NO_GEO = [
    "Crop_Year",
    "Avg_Temperature",
    "Avg_Humidity",
    "Total_Rainfall",
    "PH",
    "Clay",
    "Sand",
    "Silt",
    "Nitrogen",
    "SOC",
    "CEC"
]

CATEGORICAL_FEATURES_NO_GEO = [
    "Season"
]

In [40]:
X_train_no_geo = train_df[FEATURE_COLUMNS_NO_GEO].copy()

X_valid_no_geo = valid_df[FEATURE_COLUMNS_NO_GEO].copy()

X_test_no_geo = test_df[FEATURE_COLUMNS_NO_GEO].copy()

In [41]:
numeric_transformer_no_geo = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)
categorical_transformer_no_geo = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)
preprocessor_no_geo = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer_no_geo,
            NUMERIC_FEATURES_NO_GEO
        ),
        (
            "cat",
            categorical_transformer_no_geo,
            CATEGORICAL_FEATURES_NO_GEO
        )
    ]
)

In [42]:
extra_trees_no_geo = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_no_geo
        ),

        (
            "classifier",
            OneVsRestClassifier(
                ExtraTreesClassifier(
                    n_estimators=200,
                    random_state=42,
                    n_jobs=-1
                )
            )
        )
    ]
)
extra_trees_no_geo_result = benchmark_model(
    "ExtraTrees_No_Geography",
    extra_trees_no_geo,
    X_train_no_geo,
    Y_train,
    X_valid_no_geo,
    Y_valid
)

print(extra_trees_no_geo_result)

d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'Model': 'ExtraTrees_No_Geography', 'Micro_F1': 0.8552, 'Macro_F1': 0.6203, 'Hamming_Loss': 0.0288, 'Precision@5': np.float64(0.728), 'Training_Time_Sec': 950.28}


In [47]:
lgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            OneVsRestClassifier(
                LGBMClassifier(
                    random_state=42,
                    verbose=-1,
                    n_jobs=-1
                )
            )
        )
    ]
)
param_distributions = {

    "classifier__estimator__n_estimators":
        [100, 200, 300, 500],

    "classifier__estimator__learning_rate":
        [0.01, 0.03, 0.05, 0.1],

    "classifier__estimator__num_leaves":
        [31, 63, 127],

    "classifier__estimator__max_depth":
        [-1, 5, 10, 15],

    "classifier__estimator__subsample":
        [0.7, 0.8, 0.9, 1.0],

    "classifier__estimator__colsample_bytree":
        [0.7, 0.8, 0.9, 1.0],

    "classifier__estimator__min_child_samples":
        [10, 20, 50]
}
from sklearn.metrics import make_scorer, f1_score

micro_f1_scorer = make_scorer(
    f1_score,
    average="micro"
)

In [51]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=lgb_pipeline,

    param_distributions=param_distributions,

    n_iter=10,

    cv=2,

    verbose=2,

    random_state=42,

    n_jobs=-1,

    scoring=micro_f1_scorer
)

In [54]:
print(random_search.best_score_)
print(random_search.best_params_)
best_lgb_model = random_search.best_estimator_
best_lgb_model

0.5883792198252167
{'classifier__estimator__subsample': 0.8, 'classifier__estimator__num_leaves': 31, 'classifier__estimator__n_estimators': 300, 'classifier__estimator__min_child_samples': 20, 'classifier__estimator__max_depth': 15, 'classifier__estimator__learning_rate': 0.05, 'classifier__estimator__colsample_bytree': 0.9}


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](66,)","[ 0, 1, 2,...,63,64,65]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['State_Name','District_Name','Crop_Year',...,'Nitrogen','SOC','CEC']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder

In [52]:
random_search.fit(
    X_train,
    Y_train
)

Fitting 2 folds for each of 10 candidates, totalling 20 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...erbose=-1)))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__estimator__colsample_bytree': [0.7, 0.8, ...], 'classifier__estimator__learning_rate': [0.01, 0.03, ...], 'classifier__estimator__max_depth': [-1, 5, ...], 'classifier__estimator__min_child_samples': [10, 20, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",make_scorer(f...average=micro)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",2
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where

In [55]:
param_grid = [
    {
        "n_estimators": 200,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "max_depth": -1
    },
    {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": 15
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": 15
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "max_depth": -1
    },
    {
        "n_estimators": 700,
        "learning_rate": 0.03,
        "num_leaves": 127,
        "max_depth": -1
    }
]

In [56]:
import time
def tune_lightgbm(
    param_grid,
    preprocessor,
    X_train,
    Y_train,
    X_valid,
    Y_valid
):

    results = []

    best_model = None
    best_score = -1

    for params in param_grid:

        print("=" * 80)
        print(params)

        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),

                (
                    "classifier",
                    OneVsRestClassifier(
                        LGBMClassifier(
                            random_state=42,
                            verbose=-1,
                            n_jobs=-1,
                            subsample=0.8,
                            colsample_bytree=0.9,
                            min_child_samples=20,
                            **params
                        )
                    )
                )
            ]
        )

        start = time.time()

        pipeline.fit(
            X_train,
            Y_train
        )

        training_time = (
            time.time() - start
        )

        metrics = evaluate_model(
            pipeline,
            X_valid,
            Y_valid
        )

        result = {
            **params,
            "Micro_F1":
                metrics["micro_f1"],
            "Macro_F1":
                metrics["macro_f1"],
            "Precision@5":
                metrics["precision_at_5"],
            "Training_Time":
                training_time
        }

        results.append(result)

        if metrics["micro_f1"] > best_score:

            best_score = metrics["micro_f1"]
            best_model = pipeline

    return (
        pd.DataFrame(results)
        .sort_values(
            "Micro_F1",
            ascending=False
        ),
        best_model
    )

In [57]:
tuning_results, best_lgb_model = tune_lightgbm(
    param_grid,
    preprocessor,
    X_train,
    Y_train,
    X_valid,
    Y_valid
)

{'n_estimators': 200, 'learning_rate': 0.1, 'num_leaves': 31, 'max_depth': -1}


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 15}


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 63, 'max_depth': 15}


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'n_estimators': 500, 'learning_rate': 0.03, 'num_leaves': 63, 'max_depth': -1}


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'n_estimators': 700, 'learning_rate': 0.03, 'num_leaves': 127, 'max_depth': -1}


d:\Agrisense Application\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [58]:
tuning_results

,n_estimators,learning_rate,num_leaves,max_depth,Micro_F1,Macro_F1,Precision@5,Training_Time
4,700,0.03,127,-1,0.873740,0.634964,0.735909,625.265192
3,500,0.03,63,-1,0.872193,0.635252,0.734282,295.550524
2,500,0.05,63,15,0.871343,0.634264,0.733591,265.968537
0,200,0.10,31,-1,0.866184,0.631122,0.730744,87.306036
1,300,0.05,31,15,0.863683,0.626460,0.733266,129.054730


In [59]:
best_params = tuning_results.iloc[0]

In [62]:
train_final_df = pd.concat(
    [train_df, valid_df],
    ignore_index=True
)

In [61]:
FEATURE_COLUMNS = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season",
    "Avg_Temperature",
    "Avg_Humidity",
    "Total_Rainfall",
    "PH",
    "Clay",
    "Sand",
    "Silt",
    "Nitrogen",
    "SOC",
    "CEC"
]

In [63]:
X_train_final = train_final_df[FEATURE_COLUMNS]

Y_train_final = mlb.transform(
    train_final_df["Crop"]
)

In [64]:
X_test_final = test_df[FEATURE_COLUMNS]

Y_test_final = mlb.transform(
    test_df["Crop"]
)

In [65]:
final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            OneVsRestClassifier(
                LGBMClassifier(
                    n_estimators=700,
                    learning_rate=0.03,
                    num_leaves=127,
                    max_depth=-1,
                    subsample=0.8,
                    colsample_bytree=0.9,
                    min_child_samples=20,
                    random_state=42,
                    verbose=-1,
                    n_jobs=-1
                )
            )
        )
    ]
)
final_model.fit(
    X_train_final,
    Y_train_final
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](66,)","[ 0, 1, 2,...,63,64,65]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['State_Name','District_Name','Crop_Year',...,'Nitrogen','SOC','CEC']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder

In [66]:
final_test_metrics = evaluate_model(
    final_model,
    X_test_final,
    Y_test_final
)

print(final_test_metrics)

d:\Agrisense Application\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Agrisense Application\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'micro_f1': 0.8687510988688976, 'macro_f1': 0.6192201973643233, 'hamming_loss': 0.026628854763051348, 'precision_at_5': np.float64(0.7312929174023935)}


In [67]:
import joblib
import os

os.makedirs("artifacts", exist_ok=True)

In [68]:
joblib.dump(
    final_model,
    "artifacts/lightgbm_crop_model.pkl"
)

['artifacts/lightgbm_crop_model.pkl']

In [69]:
joblib.dump(
    mlb,
    "artifacts/mlb.pkl"
)

['artifacts/mlb.pkl']

In [70]:
loaded_model = joblib.load(
    "artifacts/lightgbm_crop_model.pkl"
)

loaded_mlb = joblib.load(
    "artifacts/mlb.pkl"
)

In [74]:
sample = X_test_final.head(1)

# pred = loaded_model.predict(sample)

# loaded_mlb.inverse_transform(pred)

In [76]:
def get_top_k_predictions(
    model,
    mlb,
    sample_df,
    k=5
):

    processed_sample = (
        model.named_steps["preprocessor"]
        .transform(sample_df)
    )

    classifier = (
        model.named_steps["classifier"]
    )

    probabilities = np.array(
        [
            estimator.predict_proba(
                processed_sample
            )[:, 1]

            for estimator in classifier.estimators_
        ]
    ).T

    top_indices = np.argsort(
        probabilities[0]
    )[-k:][::-1]

    return [
        {
            "crop": mlb.classes_[idx],
            "probability":
                round(
                    float(
                        probabilities[0][idx]
                    ) * 100,
                    2
                )
        }

        for idx in top_indices
    ]

In [77]:
sample = pd.DataFrame([
    {
        "State_Name": "KARNATAKA",
        "District_Name": "BANGALORE RURAL",
        "Crop_Year": 2025,
        "Season": "Kharif",
        "Avg_Temperature": 27.5,
        "Avg_Humidity": 78,
        "Total_Rainfall": 900,
        "PH": 6.8,
        "Clay": 28,
        "Sand": 35,
        "Silt": 37,
        "Nitrogen": 280,
        "SOC": 0.7,
        "CEC": 18
    }
])

predictions = get_top_k_predictions(
    final_model,
    mlb,
    sample,
    k=5
)

predictions

d:\Agrisense Application\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[{'crop': 'Maize', 'probability': 97.86},
 {'crop': 'Arhar/Tur', 'probability': 94.29},
 {'crop': 'Groundnut', 'probability': 90.55},
 {'crop': 'Jowar', 'probability': 81.08},
 {'crop': 'Sesamum', 'probability': 63.24}]

In [75]:
get_top_k_predictions(loaded_model,loaded_mlb,sample,5)

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: State_Name: str, District_Name: str, Season: str

In [ ]:
new_df = 